# Phase 1: Loading, Exploring and Cleaning the Datasets

In [7]:
import pandas as pd

# Loading data
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')

# Merging on TransactionID
train = pd.merge(train_trans, train_id, on='TransactionID', how='left')

print("Data successfully loaded and merged...")
print(f"Total rows: {len(train)}")
print(f"Train dataset shape: {train.shape}")

Data successfully loaded and merged...
Total rows: 590540
Train dataset shape: (590540, 434)


In [5]:
# Calculating the percentage of missing values for each feature
missing_percentages = (train.isnull().sum() / len(train)) * 100

# Sorting the results in descending order
missing_percentages = missing_percentages.sort_values(ascending=False)

# Top 30 features with most missing values
print('Top 30 features with most missing values: ')
print(missing_percentages.head(30))

Top 30 features with most missing values: 
id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_04    88.768923
id_03    88.768923
D6       87.606767
id_33    87.589494
id_09    87.312290
D8       87.312290
id_10    87.312290
D9       87.312290
id_30    86.865411
id_32    86.861855
id_34    86.824771
id_14    86.445626
V138     86.123717
V139     86.123717
V148     86.123717
dtype: float64


In [8]:
# Dropping features with more than 99% of missing data
features_to_drop = missing_percentages[missing_percentages > 99].index

# Drop those columns
train = train.drop(columns=features_to_drop)
print(f"Total features dropped: {len(features_to_drop)}")
print(f"New Train dataset shape: {train.shape}")

Total features dropped: 9
New Train dataset shape: (590540, 425)


# Phase 2: Feature Engineering


In [10]:
# Converting seconds into hours, then finding the remainder of a 24-hour cycle
train['hour_of_day'] = (train['TransactionDT'] // 3600) % 24

# Converting seconds to days to find the remainder of a 7-day cycle
train['day-of-week'] = (train['TransactionDT'] // (3600*24)) % 7

print("Time features engineered successfully...")
train

Time features engineered successfully...


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,hour_of_day,day-of-week
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0


# Converting P_emaildomain feature into an target encoded feature

In [12]:
import numpy as np
from sklearn.model_selection import KFold

# Creating an empty column to hold new target encoded values
train['P_emaildomain_target_encoded'] = np.nan

# Setting up the K-Fold splitter for 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Looping through each fold, separating the training chunk from the isolated chunk
for train_index, test_index in kf.split(train):
    # Isolating the data
    X_train_fold = train.iloc[train_index]
    X_val_fold = train.iloc[test_index]

    # Calculating the fraud probability for each domain using only the training chunk
    domain_probabilities = X_train_fold.groupby('P_emaildomain')['isFraud'].mean()

    # Apply these calculated probabilities to the isolated chunk
    train.loc[test_index, 'P_emaildomain_target_encoded'] = X_val_fold['P_emaildomain'].map(domain_probabilities)

# For rare domains, the global probability will be filled
global_mean = train['isFraud'].mean()
train['P_emaildomain_target_encoded'] = train['P_emaildomain_target_encoded'].fillna(global_mean)

# Comparision
print(train[['P_emaildomain','isFraud','P_emaildomain_target_encoded']].head(15))



    P_emaildomain  isFraud  P_emaildomain_target_encoded
0             NaN        0                      0.034990
1       gmail.com        0                      0.043825
2     outlook.com        0                      0.095816
3       yahoo.com        0                      0.022679
4       gmail.com        0                      0.043789
5       gmail.com        0                      0.043461
6       yahoo.com        0                      0.022628
7        mail.com        0                      0.178571
8   anonymous.com        0                      0.023125
9       yahoo.com        0                      0.023063
10      gmail.com        0                      0.043789
11    hotmail.com        0                      0.053149
12    verizon.net        0                      0.007752
13        aol.com        0                      0.021606
14      yahoo.com        0                      0.022795
